# Generate `capacity_comparison.html`

**Run All** (или *Run → Run All Cells*) — пересчёт и запись `simulation_results.json`.

Данные из `data/cache/` (как в [capacity_explorer.ipynb](./capacity_explorer.ipynb)).  
После выполнения откройте в браузере: `../capacity_comparison.html`

Правьте только ячейку **Настройки** ниже.

In [1]:
from pathlib import Path
import sys

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CONFIG_PATH = ROOT / 'config' / 'capacity.example.yml'
CACHE_DIR = ROOT / 'data' / 'cache'
HTML_PATH = ROOT / 'capacity_comparison.html'

## Настройки

| Параметр | Обычно не трогать |
|----------|-------------------|
| `DAYS` | Сколько дней истории, если кэша ещё нет |
| `REFRESH` | `True` только если нужно заново скачать с GitHub |
| `CLASSIFY` | `True` — sharded/single по `pr_files` в кэше + `pr_classify` в yml |
| `ROLLOUT` | Какие PR идут в sharding |
| `PERCENTILES` | Одно число или список, напр. `[50, 90, 95]` |
| `PRIMARY_PERCENTILE` | Какой перцентиль в выводах и на HTML по умолчанию |

**Автоматически:** последний файл из `data/cache/` (при augment туда же пишутся пути PR). Квоты — `config/capacity.example.yml`.

In [2]:
DAYS = 7
REFRESH = False
CLASSIFY = True
ROLLOUT = 'all eligible'
PERCENTILES = [50, 90, 95]           # или [50, 90, 95] — все попадут в JSON
PRIMARY_PERCENTILE = 90      # перцентиль по умолчанию на HTML-странице

## Сгенерировать

In [3]:
from workflow_capacity.cache import list_datasets, resolve_dataset
from workflow_capacity.export_comparison import export_from_dataset, write_comparison_payload

REPO = 'ydb-platform/ydb'
PEAK_HOURS = list(range(9, 16))  # сводные метрики; графики — полные сутки 0–23

cached = list_datasets(CACHE_DIR)
print('Cached windows:')
if cached:
    for ds in cached:
        print(f'  - {ds.path.name} ({len(ds.jobs)} jobs, {ds.since[:10]} .. {ds.until[:10]})')
else:
    print('  (пусто — скачаем с GitHub, нужен gh auth login)')

dataset = resolve_dataset(days=DAYS, repo=REPO, cache_dir=CACHE_DIR, refresh=REFRESH)
print(f'\nDataset: {dataset.path.name} ({len(dataset.jobs)} jobs, pr_files={len(dataset.pr_files)})')

print('\nExporting comparison page (~2–5 min) ...', flush=True)
payload = export_from_dataset(
    dataset,
    config_path=CONFIG_PATH,
    classify=CLASSIFY,
    rollout_label=ROLLOUT,
    peak_hours=PEAK_HOURS,
    percentiles=PERCENTILES,
    primary_percentile=PRIMARY_PERCENTILE,
)
print(f'Percentiles: {payload["meta"]["percentiles"]}, primary={payload["meta"]["primary_percentile"]}')
written = write_comparison_payload(payload, root=ROOT)
for path in written:
    print(f'  wrote {path.relative_to(ROOT)}', flush=True)

print(f"\nГотово: {len(payload['scenarios'])} scenarios, {len(payload['interactive'])} sweep points")
print(f'Откройте в браузере: file://{HTML_PATH}')

Cached windows:
  - jobs_ydb-platform_ydb_2026-06-07_2026-06-14.json.partial.json (2101 jobs, 2026-06-07 .. 2026-06-14)
cache: 7d window not found, using latest jobs_ydb-platform_ydb_2026-06-07_2026-06-14.json.partial.json (2101 jobs)

Dataset: jobs_ydb-platform_ydb_2026-06-07_2026-06-14.json.partial.json (2101 jobs, pr_files=0)

Exporting comparison page (~2–5 min) ...
[export 1/31] current ...
[export 2/31] vcpu+10% ...
[export 3/31] all+25% ...
[export 4/31] all x2 ...
[export 5/31] instances=88 ...
[export 6/31] instances=99 ...
[export 7/31] instances=121 ...
[export 8/31] instances=137 ...
[export 9/31] instances=165 ...
[export 10/31] instances=198 ...
[export 11/31] instances=220 ...
[export 12/31] vcpu=4320 ...
[export 13/31] vcpu=4860 ...
[export 14/31] vcpu=6750 ...
[export 15/31] vcpu=8100 ...
[export 16/31] vcpu=9450 ...
[export 17/31] vcpu=10800 ...
[export 18/31] ram_gb=18400 ...
[export 19/31] ram_gb=20700 ...
[export 20/31] ram_gb=25300 ...
[export 21/31] ram_gb=28750 